In [ ]:

import pandas as pd
import numpy as np

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split 
from sklearn.datasets import make_classification, make_regression


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
RANDOM_STATE=42

X_clf, y_clf = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    weights=[0.6, 0.4],
    class_sep=1.0,
    random_state=RANDOM_STATE
)

X_clf = pd.DataFrame(
    X_clf,
    columns=[f"feature_{i}" for i in range(1, 11)]
)

y_clf = pd.Series(y_clf, name="target")

X_train, X_test, y_train, y_test= train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

print("Classification dataset:", X_clf.shape)
print("Training features:", X_train_clf.shape)
print("Testing features:", X_test_clf.shape)

print("\nTraining target distribution:")
print(y_train_clf.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test_clf.value_counts(normalize=True))


base_estimator = DecisionTreeClassifier(random_state=42)

ada_clf = AdaBoostClassifier(
    estimator=base_estimator,
    random_state=42
)

param_grid_clf = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0],
    "estimator__max_depth": [1, 2, 3],
    "estimator__min_samples_leaf": [1, 2, 4]
}

scoring_clf = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted"
}

grid_clf = GridSearchCV(
    estimator=ada_clf,
    param_grid=param_grid_clf,
    scoring=scoring_clf,
    refit="f1",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_clf.fit(X_train, y_train)

print("Best Parameters:", grid_clf.best_params_)
print("Best CV F1 Score:", grid_clf.best_score_)

best_clf = grid_clf.best_estimator_

y_pred_clf = best_clf.predict(X_test)

clf_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_clf),
    "Precision": precision_score(
        y_test, y_pred_clf, average="weighted", zero_division=0
    ),
    "Recall": recall_score(
        y_test, y_pred_clf, average="weighted", zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, y_pred_clf, average="weighted", zero_division=0
    )
}

print("\nTest Set Metrics:")
print(pd.Series(clf_metrics))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_clf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_clf, zero_division=0))


Classification dataset: (2000, 10)
Training features: (1600, 10)
Testing features: (400, 10)

Training target distribution:
target
0    0.59875
1    0.40125
Name: proportion, dtype: float64

Testing target distribution:
target
0    0.6
1    0.4
Name: proportion, dtype: float64
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters: {'estimator__max_depth': 3, 'estimator__min_samples_leaf': 1, 'learning_rate': 1.0, 'n_estimators': 200}
Best CV F1 Score: 0.9436248661447273

Test Set Metrics:
Accuracy     0.922500
Precision    0.922592
Recall       0.922500
F1 Score     0.922540
dtype: float64

Confusion Matrix:
[[224  16]
 [ 15 145]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.94       240
           1       0.90      0.91      0.90       160

    accuracy                           0.92       400
   macro avg       0.92      0.92      0.92       400
weighted avg       0.92      0.92      0.

In [ ]:

clf_results = pd.DataFrame(grid_clf.cv_results_)

clf_results = clf_results[
    [
        "param_n_estimators",
        "param_learning_rate",
        "param_estimator__max_depth",
        "param_estimator__min_samples_leaf",
        "mean_test_accuracy",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_f1",
        "std_test_f1",
        "rank_test_f1"
    ]
]

clf_results.columns = [
    "n_estimators",
    "learning_rate",
    "max_depth",
    "min_samples_leaf",
    "CV_Accuracy",
    "CV_Precision",
    "CV_Recall",
    "CV_F1",
    "Std_F1",
    "Rank"
]

clf_results = clf_results.sort_values(
    by="Rank"
).reset_index(drop=True)

print(clf_results.to_string(index=False))


 n_estimators  learning_rate  max_depth  min_samples_leaf  CV_Accuracy  CV_Precision  CV_Recall    CV_F1   Std_F1  Rank
          200           1.00          3                 1     0.943750      0.944168   0.943750 0.943625 0.008070     1
          200           1.00          3                 2     0.943125      0.943563   0.943125 0.942974 0.006413     2
          100           1.00          3                 4     0.940625      0.940851   0.940625 0.940470 0.007329     3
          100           1.00          3                 1     0.939375      0.939604   0.939375 0.939260 0.007543     4
          200           1.00          3                 4     0.938750      0.939160   0.938750 0.938656 0.007482     5
          100           1.00          3                 2     0.938125      0.938444   0.938125 0.937974 0.006016     6
          200           1.00          2                 4     0.933750      0.933909   0.933750 0.933603 0.012074     7
           50           1.00          3 

In [ ]:

from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)


X_reg, y_reg = make_regression(
    n_samples=2000,
    n_features=10,
    n_informative=7,
    noise=15,
    bias=100,
    random_state=RANDOM_STATE
)

X_reg = pd.DataFrame(
    X_reg,
    columns=[f"feature_{i}" for i in range(1, 11)]
)

y_reg = pd.Series(y_reg, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Regression dataset:", X_reg.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)


base_regressor = DecisionTreeRegressor(random_state=42)

ada_reg = AdaBoostRegressor(
    estimator=base_regressor,
    loss="linear",
    random_state=42
)

param_grid_reg = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1.0],
    "estimator__max_depth": [2, 3, 4],
    "estimator__min_samples_leaf": [1, 2, 4]
}

scoring_reg = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

grid_reg = GridSearchCV(
    estimator=ada_reg,
    param_grid=param_grid_reg,
    scoring=scoring_reg,
    refit="RMSE",
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_reg.fit(X_train, y_train)

print("Best Parameters:", grid_reg.best_params_)
print("Best CV RMSE:", -grid_reg.best_score_)

best_reg = grid_reg.best_estimator_

y_pred_reg = best_reg.predict(X_test)

reg_metrics = {
    "MAE": mean_absolute_error(y_test, y_pred_reg),
    "MSE": mean_squared_error(y_test, y_pred_reg),
    "RMSE": root_mean_squared_error(y_test, y_pred_reg),
    "R2 Score": r2_score(y_test, y_pred_reg)
}

print("\nTest Set Metrics:")
print(pd.Series(reg_metrics))


Regression dataset: (2000, 10)
Training features: (1600, 10)
Testing features: (400, 10)
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters: {'estimator__max_depth': 4, 'estimator__min_samples_leaf': 2, 'learning_rate': 1.0, 'n_estimators': 200}
Best CV RMSE: 56.99525623358876

Test Set Metrics:
MAE           41.041016
MSE         3201.035929
RMSE          56.577698
R2 Score       0.891708
dtype: float64


In [ ]:

reg_results = pd.DataFrame(grid_reg.cv_results_)

reg_results = reg_results[
    [
        "param_n_estimators",
        "param_learning_rate",
        "param_estimator__max_depth",
        "param_estimator__min_samples_leaf",
        "mean_test_MAE",
        "mean_test_MSE",
        "mean_test_RMSE",
        "mean_test_R2",
        "std_test_RMSE",
        "rank_test_RMSE"
    ]
]

reg_results.columns = [
    "n_estimators",
    "learning_rate",
    "max_depth",
    "min_samples_leaf",
    "CV_MAE",
    "CV_MSE",
    "CV_RMSE",
    "CV_R2",
    "Std_RMSE",
    "Rank"
]

reg_results[["CV_MAE", "CV_MSE", "CV_RMSE"]] = (
    -reg_results[["CV_MAE", "CV_MSE", "CV_RMSE"]]
)

reg_results = reg_results.sort_values(
    by="Rank"
).reset_index(drop=True)

print(reg_results.to_string(index=False))


 n_estimators  learning_rate  max_depth  min_samples_leaf    CV_MAE       CV_MSE    CV_RMSE    CV_R2  Std_RMSE  Rank
          200           1.00          4                 2 41.598663  3252.937492  56.995256 0.887026  2.116190     1
          200           1.00          4                 4 41.704440  3259.524496  57.040247 0.886922  2.436124     2
          200           1.00          4                 1 41.950088  3260.567170  57.040686 0.886905  2.631977     3
          100           1.00          4                 1 43.742982  3486.769608  58.998318 0.879072  2.442963     4
          100           1.00          4                 4 43.449039  3492.939286  59.043437 0.878811  2.609951     5
          100           1.00          4                 2 43.548355  3500.258142  59.120873 0.878496  2.231696     6
          200           1.00          3                 2 45.554264  3843.666163  61.916962 0.866793  3.155308     7
          200           1.00          3                 1 45.964

In [ ]:

final_comparison = pd.DataFrame({
    "AdaBoost Classifier": clf_metrics,
    "AdaBoost Regressor": reg_metrics
})

print(final_comparison)


           AdaBoost Classifier  AdaBoost Regressor
Accuracy              0.922500                 NaN
Precision             0.922592                 NaN
Recall                0.922500                 NaN
F1 Score              0.922540                 NaN
MAE                        NaN           41.041016
MSE                        NaN         3201.035929
RMSE                       NaN           56.577698
R2 Score                   NaN            0.891708
